In [33]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))

In [34]:
import torch
from torch import nn

in_f, out_f = 512, 1536      # your block's qkv: n_embed -> 3*n_embed
r = 8

W = torch.randn(out_f, in_f)   # base weight, frozen
A = torch.randn(r, in_f)       # down-projection: 512 -> 8
B = torch.zeros(out_f, r)      # up-projection:   8 -> 1536

delta = B @ A
print(W.shape, delta.shape)
print("W params    :", W.numel())
print("lora params :", A.numel() + B.numel())
print("ratio       :", (A.numel() + B.numel()) / W.numel())

torch.Size([1536, 512]) torch.Size([1536, 512])
W params    : 786432
lora params : 16384
ratio       : 0.020833333333333332


In [35]:
B_rand = torch.randn(out_f, r)                       # what B looks like once trained
print("rank of B_rand @ A :", torch.linalg.matrix_rank(B_rand @ A).item())
print("rank of B @ A      :", torch.linalg.matrix_rank(delta).item())
print("max possible rank  :", min(in_f, out_f))


rank of B_rand @ A : 8
rank of B @ A      : 0
max possible rank  : 512


In [36]:
torch.linalg.matrix_rank(W)

tensor(512)

In [37]:
x = torch.randn(4, in_f)

A = nn.Parameter(torch.randn(r, in_f) * 0.01)
B = nn.Parameter(torch.zeros(out_f, r))

base_out = x @ W.T
lora_out = base_out + (x @ A.T) @ B.T
print("identical at init:", torch.equal(base_out, lora_out))

lora_out.square().mean().backward()
print("grad A magnitude:", A.grad.abs().sum().item())
print("grad B magnitude:", B.grad.abs().sum().item())


identical at init: True
grad A magnitude: 0.0
grad B magnitude: 33.716651916503906


In [38]:
A2 = nn.Parameter(torch.zeros(r, in_f))
B2 = nn.Parameter(torch.randn(out_f, r) * 0.01)

((x @ W.T) + (x @ A2.T) @ B2.T).square().mean().backward()
print("grad A2:", A2.grad.abs().sum().item())   # nonzero!
print("grad B2:", B2.grad.abs().sum().item())   # 0.0

grad A2: 19.29391098022461
grad B2: 0.0


In [42]:
x = torch.randn(1000, in_f)

for r_ in (2, 8, 32, 128):
    A_ = torch.randn(r_, in_f)    # what A looks like at init
    B_ = torch.randn(out_f, r_) / r_**0.5      # what B might look like once trained
    out = (x @ A_.T) @ B_.T
    print(f"r={r_:>3}  raw std {out.std():.3f}   (alpha/r)·out std {(16 / r_ * out).std():.3f}")


r=  2  raw std 23.364   (alpha/r)·out std 186.909
r=  8  raw std 22.551   (alpha/r)·out std 45.102
r= 32  raw std 22.622   (alpha/r)·out std 11.311
r=128  raw std 22.687   (alpha/r)·out std 2.836


In [40]:
import math

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale


In [41]:
base = nn.Linear(in_f, out_f, bias=False)
lora = LoRALinear(base, r=8, alpha=16)

x = torch.randn(4, in_f)
print("no-op at init:", torch.equal(base(x), lora(x)))

trainable = [(n, p.numel()) for n, p in lora.named_parameters() if p.requires_grad]
print(trainable)


no-op at init: True
[('A', 4096), ('B', 12288)]
